# Sprint 3 en Colab: TA-14b (detector de pelota) y TA-37 (tiempo y memoria)

Antes de empezar: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)**.

1. Se clona la rama ModeloAI y se aplican los patches del Sprint 3 (`sprint3-modelo.zip`).
2. Se instalan las dependencias y se descargan los modelos, también TrackNet.
3. Se suben los highlights (h1.mp4, h4.mp4…).
4. **TA-14b:** se compara YOLO con TrackNet: pelota vista, botes frente a los marcados a mano, saques, puntos y tiempo.
5. **TA-37:** se mide tiempo por minuto y memoria máxima con un vídeo de 30 s y otro de 10 min.
6. Se descargan los informes (`.md` para Jira y la PR, `.json` con los datos).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Código: rama ModeloAI + patches del Sprint 3
Sube `sprint3-modelo.zip` cuando lo pida.

In [ ]:
%cd /content
!rm -rf Proyecto_Carrera_III_Tenis sprint3-modelo
!git clone -q -b ModeloAI https://github.com/mort1zr/Proyecto_Carrera_III_Tenis.git
from google.colab import files

subido = files.upload()  # sprint3-modelo.zip
!unzip -q -o sprint3-modelo.zip
%cd /content/Proyecto_Carrera_III_Tenis
!for p in /content/sprint3-modelo/00*.patch; do git -c user.name=colab -c user.email=colab@example.com am -q --whitespace=nowarn "$p" || echo "FALLA $p"; done
!git log --oneline -3

## 2. Dependencias y modelos

In [ ]:
%cd /content/Proyecto_Carrera_III_Tenis/model
# Colab ya trae PyTorch con GPU para su versión de Python (3.13): se usa ese y se instala el resto.
# Las versiones fijadas de torch y torchvision son las de los PCs del equipo (Python 3.11).
!grep -v -E '^(torch|torchvision)==' requirements.txt > /tmp/requisitos_colab.txt
!pip install -q -r /tmp/requisitos_colab.txt
!python -c "import torch, torchvision, ultralytics, pandas; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| torchvision', torchvision.__version__, '| ultralytics', ultralytics.__version__, '| pandas', pandas.__version__)"
!python scripts/download_models.py --tracknet
!ls -la models input_videos

In [ ]:
!python -m pytest -q

## 3. Highlights
Sube los vídeos (h1.mp4, h2.mp4, h3.mp4, h4.mp4). Se guardan en `input_videos/`.

In [ ]:
import shutil

from google.colab import files

for nombre in files.upload():
    shutil.move(nombre, f"input_videos/{nombre}")
!ls -la input_videos

## 4. TA-14b: YOLO frente a TrackNet

Para cada vídeo y detector: % de fotogramas con pelota, botes frente a los marcados a mano
(`eval/labels/botes_a_mano.json`: vídeo de ejemplo, h1 y h4), saques, puntos y segundos de proceso
por minuto de vídeo (sin caché: se mide la detección de verdad).

In [ ]:
import glob

videos = ["input_videos/input_video.mp4"] + sorted(glob.glob("input_videos/h*.mp4"))
lista_videos = " ".join(videos)
print(lista_videos)
!python scripts/compare_ball_detectors.py --device cuda:0 --videos {lista_videos} --output output_videos/comparacion_pelota

## 5. TA-37: tiempo y memoria

Vídeos de 30 s y 10 min (repitiendo el de ejemplo, 1080p). Criterios de PCIT-146: con 10 min, menos
de 4 GB de memoria, y la memoria no crece con la duración. Tarda unos 20-40 min en total.

In [ ]:
!python scripts/measure_performance.py --device cuda:0 --minutes 0.5 10 --output output_videos/rendimiento_gpu

Opcional: lo mismo en CPU, solo con 30 s (en CPU, 10 min tardarían horas).

In [ ]:
!python scripts/measure_performance.py --device cpu --minutes 0.5 --output output_videos/rendimiento_cpu

## 6. Descargar los informes

In [ ]:
from google.colab import files

for informe in glob.glob("output_videos/comparacion_pelota.*") + glob.glob("output_videos/rendimiento_*.*"):
    files.download(informe)